# Fashion-MNIST Benchmark

Self-contained Fashion-MNIST benchmark with automatic dataset download, SW-KAN baselines, evaluation, and visualizations.

In [ ]:
"""
fashion_mnist_benchmark_complete.py
====================================
Complete self-contained Fashion-MNIST benchmark with automatic download.
Run: python3 fashion_mnist_benchmark_complete.py
"""

import time
import gzip
import io
import urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)


# ============================================================================ #
# 1. DATA LOADER (Fashion-MNIST with automatic download)
# ============================================================================ #

LABELS = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

# URLs for Fashion-MNIST
BASE_URL = "http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/"
FILES = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz",
}


def download_file(url):
    """Download a file from URL and return as bytes."""
    with urllib.request.urlopen(url) as response:
        return response.read()


def _load_images_from_bytes(data_bytes):
    """Load images from gzipped bytes."""
    with gzip.open(io.BytesIO(data_bytes), "rb") as f:
        data = np.frombuffer(f.read(), dtype=np.uint8, offset=16)
    return data.reshape(-1, 28 * 28).astype(np.float32)


def _load_labels_from_bytes(data_bytes):
    """Load labels from gzipped bytes."""
    with gzip.open(io.BytesIO(data_bytes), "rb") as f:
        data = np.frombuffer(f.read(), dtype=np.uint8, offset=8)
    return data.astype(np.int64)


def load_fashion_mnist(flatten_range=(-1.0, 1.0)):
    """Download and load Fashion-MNIST as torch tensors."""
    print("Downloading Fashion-MNIST...")
    
    train_images_bytes = download_file(BASE_URL + FILES["train_images"])
    train_labels_bytes = download_file(BASE_URL + FILES["train_labels"])
    test_images_bytes = download_file(BASE_URL + FILES["test_images"])
    test_labels_bytes = download_file(BASE_URL + FILES["test_labels"])
    
    print("Loading data...")
    Xtr = _load_images_from_bytes(train_images_bytes)
    ytr = _load_labels_from_bytes(train_labels_bytes)
    Xte = _load_images_from_bytes(test_images_bytes)
    yte = _load_labels_from_bytes(test_labels_bytes)

    lo, hi = flatten_range
    Xtr = Xtr / 255.0 * (hi - lo) + lo
    Xte = Xte / 255.0 * (hi - lo) + lo

    return (
        torch.from_numpy(Xtr),
        torch.from_numpy(ytr),
        torch.from_numpy(Xte),
        torch.from_numpy(yte),
    )


# ============================================================================ #
# 2. SW-KAN MODEL
# ============================================================================ #

import math

class StieltjesWigertKANLayer(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        degree: int = 6,
        gamma: float = 2.0,
        iota: float = 1.0,
        q_min: float = 0.35,
        q_max: float = 0.95,
        base_scale: float = 1.0,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.degree = degree
        self.gamma = gamma
        self.iota = iota
        self.q_min = q_min
        self.q_max = q_max

        self.pre_scale = nn.Parameter(torch.ones(in_features))
        self.pre_shift = nn.Parameter(torch.zeros(in_features))

        fan_in = in_features * (degree + 1)
        self.coeffs = nn.Parameter(
            torch.randn(out_features, in_features, degree + 1) / math.sqrt(fan_in)
        )

        self.q_raw = nn.Parameter(torch.zeros(out_features, in_features))
        self.base_weight = nn.Parameter(
            torch.full((out_features, in_features), base_scale) / math.sqrt(in_features)
        )
        self.bias = nn.Parameter(torch.zeros(out_features))

    def domain_map(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pre_scale * x + self.pre_shift
        return torch.exp(self.gamma * torch.tanh(x / self.iota))

    def edge_q(self) -> torch.Tensor:
        s = torch.sigmoid(self.q_raw)
        return self.q_min + (self.q_max - self.q_min) * s

    def sw_basis(self, z: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
        batch = z.shape[0]
        device, dtype = z.device, z.dtype
        eps = 1e-8

        z_b = z.unsqueeze(1)
        log_q = torch.log(q).unsqueeze(0)
        q_val = torch.exp(log_q)

        def b_n(n: int) -> torch.Tensor:
            return torch.exp(-(2 * n + 1.5) * log_q) * (1.0 + q_val - torch.exp((n + 1) * log_q))

        def a2_n(n: int) -> torch.Tensor:
            if n == 0:
                return torch.zeros_like(q_val)
            return torch.exp(-4 * n * log_q) * (1.0 - torch.exp(n * log_q))

        T_prev = torch.zeros(batch, self.out_features, self.in_features, device=device, dtype=dtype)
        T_curr = torch.ones(batch, self.out_features, self.in_features, device=device, dtype=dtype)
        T_list = [T_curr]

        a_curr = torch.zeros(1, self.out_features, self.in_features, device=device, dtype=dtype)

        for n in range(self.degree):
            a_next = torch.sqrt(a2_n(n + 1) + eps)
            bn = b_n(n)
            T_next = ((z_b - bn) * T_curr - a_curr * T_prev) / a_next
            T_list.append(T_next)
            T_prev, T_curr = T_curr, T_next
            a_curr = a_next

        return torch.stack(T_list, dim=-1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.domain_map(x)
        q = self.edge_q()
        S = self.sw_basis(z, q)

        poly = torch.einsum("boin,oin->bo", S, self.coeffs)
        base = F.silu(x) @ self.base_weight.t()

        return poly + base + self.bias


class SWKAN(nn.Module):
    def __init__(self, layer_sizes: list[int], degree: int = 6, gamma: float = 2.0, iota: float = 1.0, q_min: float = 0.35, q_max: float = 0.95):
        super().__init__()
        assert len(layer_sizes) >= 2
        self.layers = nn.ModuleList(
            [
                StieltjesWigertKANLayer(
                    layer_sizes[i],
                    layer_sizes[i + 1],
                    degree=degree,
                    gamma=gamma,
                    iota=iota,
                    q_min=q_min,
                    q_max=q_max,
                )
                for i in range(len(layer_sizes) - 1)
            ]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x


# ============================================================================ #
# 3. B-SPLINE KAN MODEL (Standard KAN)
# ============================================================================ #

def cox_de_boor_basis(x: torch.Tensor, grid: torch.Tensor, order: int) -> torch.Tensor:
    x = x.unsqueeze(-1)
    grid = grid.unsqueeze(0)

    basis = ((x >= grid[:, :, :-1]) & (x < grid[:, :, 1:])).to(x.dtype)

    for k in range(1, order + 1):
        left_num = x - grid[:, :, : -(k + 1)]
        left_den = grid[:, :, k:-1] - grid[:, :, : -(k + 1)]
        left = left_num / (left_den + 1e-8) * basis[:, :, :-1]

        right_num = grid[:, :, k + 1:] - x
        right_den = grid[:, :, k + 1:] - grid[:, :, 1:-k]
        right = right_num / (right_den + 1e-8) * basis[:, :, 1:]

        basis = left + right

    return basis


class BSplineKANLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int, grid_size: int = 5, order: int = 3, grid_range: tuple[float, float] = (-3.0, 3.0)):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.order = order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = torch.arange(-order, grid_size + order + 1, dtype=torch.float32) * h + grid_range[0]
        grid = grid.unsqueeze(0).repeat(in_features, 1)
        self.register_buffer("grid", grid)

        num_basis = grid_size + order
        self.spline_weight = nn.Parameter(
            torch.randn(out_features, in_features, num_basis) * (1.0 / (in_features * num_basis) ** 0.5)
        )
        self.base_weight = nn.Parameter(torch.randn(out_features, in_features) / (in_features ** 0.5))
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_clamped = x.clamp(self.grid[0, self.order].item(), self.grid[0, -self.order - 1].item())
        basis = cox_de_boor_basis(x_clamped, self.grid, self.order)
        spline_out = torch.einsum("bik,oik->bo", basis, self.spline_weight)
        base_out = F.silu(x) @ self.base_weight.t()
        return spline_out + base_out + self.bias


class BSplineKAN(nn.Module):
    def __init__(self, layer_sizes: list[int], grid_size: int = 5, order: int = 3, grid_range=(-3.0, 3.0)):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                BSplineKANLayer(layer_sizes[i], layer_sizes[i + 1], grid_size=grid_size, order=order, grid_range=grid_range)
                for i in range(len(layer_sizes) - 1)
            ]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x


# ============================================================================ #
# 4. PCA HELPER
# ============================================================================ #

def pca_fit(X_train, n_components, fit_subsample=8000):
    X_train_np = X_train.numpy()
    idx = np.random.choice(len(X_train_np), size=min(fit_subsample, len(X_train_np)), replace=False)
    fit_data = X_train_np[idx]
    mean = fit_data.mean(axis=0, keepdims=True)
    fit_centered = fit_data - mean

    U, S, Vt = np.linalg.svd(fit_centered, full_matrices=False)
    components = Vt[:n_components]
    return mean, components


def pca_transform(X, mean, components, std=None):
    X_np = X.numpy()
    reduced = (X_np - mean) @ components.T
    if std is None:
        std = reduced.std(axis=0, keepdims=True) + 1e-6
        return torch.from_numpy((reduced / std).clip(-3, 3)).float(), std
    return torch.from_numpy((reduced / std).clip(-3, 3)).float()


# ============================================================================ #
# 5. TRAINING HELPERS
# ============================================================================ #

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def evaluate(model, X, y, batch_size=1000):
    model.eval()
    correct = 0
    for start in range(0, len(X), batch_size):
        xb, yb = X[start:start + batch_size], y[start:start + batch_size]
        logits = model(xb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
    model.train()
    return correct / len(X)


def train_classifier(model, X_train, y_train, X_val, y_val, X_test, y_test, epochs, lr, batch_size=128, clip=5.0, weight_decay=0.0):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()

    n = X_train.shape[0]
    history = []
    best_val = -1.0
    best_state = None
    best_epoch = -1
    t0 = time.time()

    for epoch in range(epochs):
        perm = torch.randperm(n)
        total_loss = 0.0
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            xb, yb = X_train[idx], y_train[idx]
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            if clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            total_loss += loss.item() * len(idx)
        sched.step()
        train_loss = total_loss / n

        val_acc = evaluate(model, X_val, y_val)
        if val_acc > best_val:
            best_val = val_acc
            best_epoch = epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 5 == 0 or epoch == epochs - 1:
            history.append((epoch, train_loss, val_acc))
            print(f"    epoch {epoch:3d}  train_loss {train_loss:.4f}  val_acc {val_acc:.4f}")

    elapsed = time.time() - t0
    model.load_state_dict(best_state)
    final_test_acc = evaluate(model, X_test, y_test)
    print(f"    -> best val_acc {best_val:.4f} at epoch {best_epoch}; test_acc: {final_test_acc:.4f}")
    return final_test_acc, elapsed, history


# ============================================================================ #
# 6. MAIN
# ============================================================================ #

if __name__ == "__main__":
    import math

    print("=" * 60)
    print("Fashion-MNIST Benchmark: SW-KAN vs B-spline KAN")
    print("=" * 60)
    
    Xtr_full, ytr_full, Xte_full, yte_full = load_fashion_mnist(flatten_range=(-1, 1))

    N_TRAIN, N_VAL, N_TEST, N_COMPONENTS = 10000, 1500, 2000, 40
    all_train_idx = np.random.choice(len(Xtr_full), N_TRAIN + N_VAL, replace=False)
    train_idx, val_idx = all_train_idx[:N_TRAIN], all_train_idx[N_TRAIN:]
    test_idx = np.random.choice(len(Xte_full), N_TEST, replace=False)

    X_train_raw = Xtr_full[train_idx]
    y_train = ytr_full[train_idx]
    X_val_raw = Xtr_full[val_idx]
    y_val = ytr_full[val_idx]
    X_test_raw = Xte_full[test_idx]
    y_test = yte_full[test_idx]

    print(f"Using {N_TRAIN} train, {N_VAL} validation, {N_TEST} test samples")
    
    mean, components = pca_fit(X_train_raw, n_components=N_COMPONENTS)
    X_train, std = pca_transform(X_train_raw, mean, components)
    X_val = pca_transform(X_val_raw, mean, components, std=std)
    X_test = pca_transform(X_test_raw, mean, components, std=std)
    print(f"PCA-reduced features: {N_COMPONENTS} dims")

    sizes = [N_COMPONENTS, 48, 10]
    results = {}

    print("\n" + "-" * 60)
    print("Training SW-KAN ...")
    print("-" * 60)
    sw_model = SWKAN(sizes, degree=3)
    sw_acc, sw_time, sw_hist = train_classifier(
        sw_model, X_train, y_train, X_val, y_val, X_test, y_test,
        epochs=30, lr=1e-2, weight_decay=1e-4
    )
    results["SW-KAN (ours)"] = (sw_model, sw_acc, sw_time, sw_hist)

    print("\n" + "-" * 60)
    print("Training Standard KAN (B-spline) ...")
    print("-" * 60)
    bsp_model = BSplineKAN(sizes, grid_size=5, order=3, grid_range=(-3.0, 3.0))
    bsp_acc, bsp_time, bsp_hist = train_classifier(
        bsp_model, X_train, y_train, X_val, y_val, X_test, y_test,
        epochs=30, lr=1e-2, weight_decay=1e-4
    )
    results["Standard KAN (B-spline)"] = (bsp_model, bsp_acc, bsp_time, bsp_hist)

    print("\n" + "=" * 66)
    print(f"{'Model':<26}{'Params':>8}{'Test Acc':>12}{'Time (s)':>12}")
    print("-" * 66)
    for name, (model, acc, elapsed, hist) in results.items():
        print(f"{name:<26}{count_params(model):>8}{acc:>12.4f}{elapsed:>12.1f}")

    # Plot results
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    names = list(results.keys())
    accs = [results[n][1] for n in names]
    params = [count_params(results[n][0]) for n in names]
    colors = ["#2E86AB", "#A23B72"]

    # Accuracy bar chart
    axes[0].bar(names, accs, color=colors)
    for i, (a, p) in enumerate(zip(accs, params)):
        axes[0].text(i, a + 0.01, f"{a:.3f}\n({p:,} params)", ha="center", fontsize=9)
    axes[0].set_ylabel("Test accuracy")
    axes[0].set_ylim(0, 1.05)
    axes[0].set_title("Fashion-MNIST test accuracy (PCA-40 features)")
    axes[0].tick_params(axis="x", rotation=15)

    # Training curves
    for name, color in zip(names, colors):
        hist = results[name][3]
        epochs_h = [h[0] for h in hist]
        accs_h = [h[2] for h in hist]
        axes[1].plot(epochs_h, accs_h, "-o", color=color, label=name, markersize=3, linewidth=2)
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("validation accuracy")
    axes[1].set_title("Validation accuracy over training")
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    
    # Display plot inline (for notebook) or save
    try:
        from IPython.display import display
        display(fig)
    except:
        plt.savefig("fashion_mnist_comparison.png", dpi=150, bbox_inches="tight")
        print("\nSaved plot to fashion_mnist_comparison.png")
    
    plt.show()
    
    # Print summary
    print("\n" + "=" * 66)
    print("SUMMARY RESULTS")
    print("=" * 66)
    print(f"{'Model':<26} | {'Parameters':>10} | {'Test Accuracy':>14} | {'Time (s)':>10}")
    print("-" * 66)
    for name, (model, acc, elapsed, hist) in results.items():
        print(f"{name:<26} | {count_params(model):>10,} | {acc:>14.4f} | {elapsed:>10.1f}")